#### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import KMRF class
from kmrf import KMRF
from KMRF_training_config import *

# Import parallelization tools
from joblib import Parallel, delayed
import multiprocessing

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Get number of CPUs
n_cpus = multiprocessing.cpu_count()
n_jobs = max(1, n_cpus // 2)  # Use half of available CPUs

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")
print(f"  Available CPUs: {n_cpus}")
print(f"  Using {n_jobs} CPUs for parallel training")

✓ Libraries imported successfully
  Pandas version: 2.3.3
  NumPy version: 2.3.4
  Available CPUs: 8
  Using 4 CPUs for parallel training


## Getting Price Data

#### -------------------------------------------------------------------------------------------------

In [ ]:
# Prepare data
international_index_symbol_names = pd.read_csv('data/inputs/fmp_index_list.csv').set_index('symbol')['name']
international_index_symbol_names = international_index_symbol_names[~international_index_symbol_names.index.isin(['^GSPC', '^NDX'])].to_dict()
commodity_symbol_names = pd.read_csv('data/inputs/fmp_commodity_list.csv').set_index('symbol')['name'].to_dict()
etf_symbol_names = {
    # BOND ETFS
    'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    'IEF': 'iShares 7-10 Year Treasury Bond ETF',
    # International EQUITY ETFS
    'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    '^RUI': 'Russell 1000',
    '^RUA': 'Russell 3000',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    'VOO': 'Vanguard S&P 500 ETF',
    'RSP': 'Invesco S&P 500 Equal Weight ETF',
    'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'QQQM': 'Invesco Nasdaq 100 ETF',
    'ONEQ': 'Fidelity Nasdaq Composite Index ETF',
    'IWM': 'iShares Russell 2000 ETF',
    'IWB': 'iShares Russell 1000 ETF',
    'IWV': 'iShares Russell 3000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS (SELECT SECTOR SPDRS)
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    'XLRE': 'Real Estate Select Sector SPDR',
    'XLC': 'Communication Services Select Sector SPDR',
    
    # GROWTH ETFs
    'IVW': 'iShares S&P 500 Growth ETF',
    'VONG': 'Vanguard Russell 1000 Growth ETF',
    'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    'VUG': 'Vanguard Growth ETF',
    'SPYG': 'SPDR Portfolio S&P 500 Growth ETF',
    
    # VALUE ETFs
    'IVE': 'iShares S&P 500 Value ETF',
    'VONV': 'Vanguard Russell 1000 Value ETF',
    'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    'VTV': 'Vanguard Value ETF',
    'SPYV': 'SPDR Portfolio S&P 500 Value ETF',
    
    # SIZE ETFs
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    'IJH': 'iShares Core S&P Mid-Cap ETF',
    'IJR': 'iShares Core S&P Small-Cap ETF',
    'MDY': 'SPDR S&P MidCap 400 ETF',
    'SLY': 'SPDR S&P 600 Small Cap ETF',
    'VO': 'Vanguard Mid-Cap ETF',
    'VB': 'Vanguard Small-Cap ETF',
    'SCHA': 'Schwab U.S. Small-Cap ETF',
    'SCHM': 'Schwab U.S. Mid-Cap ETF',
    'VTWO': 'Vanguard Russell 2000 ETF',
    'VTHR': 'Vanguard Russell 3000 ETF',
    'THRK': 'iShares Russell 3000 ETF',
    'SPSM': 'SPDR Portfolio S&P 600 Small Cap ETF',
    'SMLF': 'iShares Small-Cap US Equity Factor ETF',
    
    # NASDAQ SPECIFIC
    'QTEC': 'First Trust Nasdaq-100 Technology Sector Index Fund',
    'QQEW': 'First Trust Nasdaq-100 Equal Weighted Index Fund',
    'QQQG': 'Pacer Nasdaq 100 Top 50 Cash Cows Dividend Growth ETF',
    'QQQV': 'Pacer Nasdaq 100 Top 50 Value ETF',
    
    # DIVIDEND/QUALITY
    'SCHD': 'Schwab U.S. Dividend Equity ETF',
    'VYM': 'Vanguard High Dividend Yield ETF',
    'DVY': 'iShares Select Dividend ETF',
    'QUAL': 'iShares MSCI USA Quality Factor ETF',
    'USMV': 'iShares MSCI USA Min Vol Factor ETF',
    
    # EQUAL WEIGHT
    'EWSC': 'Invesco S&P SmallCap 600 Equal Weight ETF',
    'EWMC': 'Invesco S&P MidCap 400 Equal Weight ETF',
}
universe_symbol_names = {
    'IVV': 'IVV - iShares Core S&P 500 ETF',
    'IJH': 'IJH - iShares Core S&P Mid-Cap ETF',
    'IWM': 'IWM - iShares Russell 2000 ETF',
    'EFA': 'EFA - iShares MSCI EAFE ETF',
    'EEM': 'EEM - iShares MSCI Emerging Markets ETF',
    'AGG': 'AGG - iShares Core U.S. Aggregate Bond ETF',
    'SPTL': 'SPTL - SPDR Portfolio Long Term Treasury ETF',
    'HYG': 'HYG - iShares iBoxx $ High Yield Corporate Bond ETF',
    'SPBO': 'SPBO - SPDR Portfolio Corporate Bond ETF',
    'IYR': 'IYR - iShares U.S. Real Estate ETF',
    'DBC': 'DBC - Invesco DB Commodity Index Tracking Fund',
    'GLD': 'GLD - SPDR Gold Shares',
}

# international_index_data = pd.read_csv('data/processed/index_data.csv', index_col=0, header=[0, 1], parse_dates=True)
commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0, 1], parse_dates=True)
etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
universe_data = pd.read_csv('data/processed/universe_etfs.csv', index_col=0, header=[0, 1], parse_dates=True)

# international_index_close_cols = international_index_data.columns[international_index_data.columns.get_level_values(1) == 'close']
# international_index_close_prices = international_index_data[international_index_close_cols].droplevel(1, axis=1).rename(columns=international_index_symbol_names)
# international_index_close_prices = international_index_close_prices[international_index_close_prices.columns[international_index_close_prices.columns.isin(['S&P 500', 'NASDAQ 100']) == False]]
# international_index_close_prices.columns = [col.replace('/', ' ') for col in international_index_close_prices.columns]

commodity_data_close_cols = commodity_data.columns[commodity_data.columns.get_level_values(1) == 'close']
commodity_close_prices = commodity_data[commodity_data_close_cols].droplevel(1, axis=1).rename(columns=commodity_symbol_names)
commodity_close_prices.columns = [col.replace('/', ' ') for col in commodity_close_prices.columns]

etf_close_cols = etf_data.columns[etf_data.columns.get_level_values(1) == 'close']
etf_close_prices = etf_data[etf_close_cols].droplevel(1, axis=1).rename(columns=etf_symbol_names)

universe_close_cols = universe_data.columns[universe_data.columns.get_level_values(1) == 'close']
universe_close_prices = universe_data[universe_close_cols].droplevel(1, axis=1).rename(columns=universe_symbol_names)

print('Commodities:', commodity_close_prices.columns.tolist())
print('ETFs:', etf_close_prices.columns.tolist())
print('Universe:', universe_close_prices.columns.tolist())

In [ ]:
etf_always_disclude = ['Vanguard S&P 500 ETF', 'Real Estate Select Sector SPDR', 'Communication Services Select Sector SPDR']
etf_disclude = [name for name in etf_data.columns if 'Russell 1000' in name or 'Russell 3000' in name]\
                        + ['S&P 500', 'Nasdaq Composite', 'Dow Jones Industrial Average', 'Nasdaq 100', 'Russell 2000']

etf_include = list(set(etf_close_prices.columns.tolist()) - set(etf_always_disclude) - set(etf_disclude))
etf_close_prices = etf_close_prices[etf_include]

us_treasury = ['SPDR Bloomberg 1-3 Month T-Bill ETF', 'iShares 1-3 Year Treasury Bond ETF', 'iShares 7-10 Year Treasury Bond ETF']
int_equity = ['Vanguard Total International Stock ETF', 'Vanguard FTSE Developed Markets ETF',\
                                        'Vanguard FTSE Emerging Markets ETF','Vanguard FTSE Europe ETF',\
                                        'Vanguard FTSE Pacific ETF', 'iShares China Large-Cap ETF',\
                                        'iShares MSCI Japan ETF', 'iShares MSCI India ETF']
us_equity = list(set(etf_include) - set(us_treasury) - set(int_equity))
print('US Equity:', us_equity)
print('US Bond:', us_treasury)
print('US Traded International Equity ETFs:', int_equity)

#### -------------------------------------------------------------------------------------------------

In [ ]:
# Create DataFrame of asset names
asset_names_df = pd.DataFrame({
    'universe': get_assets_by_class('universe') + ['']*7,
    'us_equity': get_assets_by_class('us_equity'),
    'commodity': get_assets_by_class('commodity') + ['']*5,
    'int_equity': get_assets_by_class('int_equity') + ['']*11,

})

from pandas import option_context
with option_context('display.max_colwidth', None):
    display(asset_names_df)

In [ ]:
df = etf_close_prices['SPDR S&P 500 ETF'].to_frame().dropna()
rebal_dates = df.loc['2018-12-31':].index[::21]
rebal_dates

## Batch Train KMRF Model

In [3]:
def train_single_asset(asset_name, rebal_date):
    """Train KMRF model for a single asset."""
    try:
        print(f"Starting training for {asset_name}...")
        
        TRAINING_CONFIG = KMRF_Training_Config(
            asset_name=asset_name,
            classification_type='original',
            use_data_type='master',
            end_date=rebal_date,
            feature_window_size=1,
            feature_asset_classes=[],
            cross_asset_specific=[],  # empty means all (only referring to universe assets)
            use_boruta_selection=True,
            use_consensus_selection=False
        )

        model = KMRF(
            asset_class=TRAINING_CONFIG.get_asset_class(),
            asset_name=TRAINING_CONFIG.get_asset_name(),
            classification_type=TRAINING_CONFIG.get_classification_type(),
            end_date=TRAINING_CONFIG.get_date_ranges()['end_date'],
            use_data_type=TRAINING_CONFIG.get_use_data_type(),
            feature_window_size=TRAINING_CONFIG.get_feature_window_size(),  
            feature_asset_classes=TRAINING_CONFIG.get_cross_asset_features(),
            cross_asset_specific=TRAINING_CONFIG.get_cross_asset_specific(),
            xgb_params=TRAINING_CONFIG.get_xgb_params(),
            use_boruta_selection=TRAINING_CONFIG.get_use_boruta_selection(),
            use_consensus_selection=TRAINING_CONFIG.get_use_consensus_selection(),
        )

        model.pipeline(optimize=False)

        path = f'saved_models/KMRF_new/{model.classification_type}/{model.asset_class}/'
        path += f'{model.asset_name}_KMRF_model.pkl'
        model.save_model(path)
        
        print(f"✓ Completed training for {asset_name}")
        return {'asset_name': asset_name, 'status': 'success', 'path': path}
        
    except Exception as e:
        print(f"✗ Error training {asset_name}: {str(e)}")
        return {'asset_name': asset_name, 'status': 'failed', 'error': str(e)}

rebal_date = '20181231'
# Parallel training
for asset_class in ['us_equity']:
    assets = [name for name in asset_names_df[asset_class].tolist() if name]  # Remove empty strings
    
    print(f"\n{'='*80}")
    print(f"TRAINING {len(assets)} ASSETS IN PARALLEL ({n_jobs} workers)")
    print(f"{'='*80}\n")
    
    # Run parallel training
    results = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(train_single_asset)(asset_name, rebal_date) 
        for asset_name in assets
    )
    
    # Summary
    successful = sum(1 for r in results if r['status'] == 'success')
    failed = sum(1 for r in results if r['status'] == 'failed')
    
    print(f"\n{'='*80}")
    print(f"TRAINING COMPLETE")
    print(f"{'='*80}")
    print(f"✓ Successful: {successful}/{len(assets)}")
    print(f"✗ Failed: {failed}/{len(assets)}")
    
    if failed > 0:
        print("\nFailed assets:")
        for r in results:
            if r['status'] == 'failed':
                print(f"  - {r['asset_name']}: {r['error']}")


TRAINING 19 ASSETS IN PARALLEL (4 workers)



[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


Starting training for SPDR S&P 500 ETF...Starting training for SPDR Dow Jones Industrial Average ETF...

Starting training for Invesco QQQ Trust...
Starting training for iShares Russell 2000 ETF...
KMRF model initialized
  Asset: SPDR Dow Jones Industrial Average ETF
  Asset class: us_equity
  Classification type: original
  End date: 20181231
  Data type: master
  Data path: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/data/master_df.csv
  KAMA+MSR model directory: /Users/jessegoodman/Desktop/Stevens MFE/Sem3/FE800/FE800_project_code/saved_models/KAMA_MSR/us_equity/20181231
  Validation period: 2019-01-02 to 2021-12-31
  Test start: 2022-01-02
  Random seed: 1010
  Feature window size: 1 days
  Feature asset classes: []
KMRF model initialized
  Asset: SPDR S&P 500 ETF
  Feature selection: Boruta=True, Consensus=False  Asset class: us_equity

  Classification type: original
  End date: 20181231
  Data type: masterKMRF model initialized

  Asset: iShares Russell

[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:  9.8min


Iteration: 	49 / 100
Confirmed: 	114
Tentative: 	6
Rejected: 	2
Iteration: 	42 / 100
Confirmed: 	114
Tentative: 	4
Rejected: 	4
Iteration: 	42 / 100
Confirmed: 	114
Tentative: 	4
Rejected: 	4
Iteration: 	21 / 100
Confirmed: 	113
Tentative: 	7
Rejected: 	2
Iteration: 	21 / 100
Confirmed: 	113
Tentative: 	7
Rejected: 	2
Iteration: 	50 / 100
Confirmed: 	114
Tentative: 	6
Rejected: 	2
Iteration: 	50 / 100
Confirmed: 	114
Tentative: 	6
Rejected: 	2
Iteration: 	43 / 100
Confirmed: 	114
Tentative: 	4
Rejected: 	4
Iteration: 	43 / 100
Confirmed: 	114
Tentative: 	4
Rejected: 	4
Iteration: 	22 / 100
Confirmed: 	113
Tentative: 	7
Rejected: 	2
Iteration: 	22 / 100
Confirmed: 	113
Tentative: 	7
Rejected: 	2
Iteration: 	51 / 100
Confirmed: 	114
Tentative: 	6
Rejected: 	2
Iteration: 	51 / 100
Confirmed: 	114
Tentative: 	6
Rejected: 	2
Iteration: 	44 / 100
Confirmed: 	114
Tentative: 	4
Rejected: 	4
Iteration: 	44 / 100
Confirmed: 	114
Tentative: 	4
Rejected: 	4
Iteration: 	23 / 100
Confirmed: 	113
Ten

...

[Parallel(n_jobs=4)]: Done  14 out of  19 | elapsed: 25.6min remaining:  9.1min


Iteration: 	98 / 100
Confirmed: 	108
Tentative: 	7
Rejected: 	7
Iteration: 	37 / 100
Confirmed: 	115
Tentative: 	4
Rejected: 	3
Iteration: 	37 / 100
Confirmed: 	115
Tentative: 	4
Rejected: 	3
Iteration: 	80 / 100
Confirmed: 	109
Tentative: 	8
Rejected: 	5
Iteration: 	80 / 100
Confirmed: 	109
Tentative: 	8
Rejected: 	5
Iteration: 	99 / 100
Confirmed: 	108
Tentative: 	7
Rejected: 	7


BorutaPy finished running.

Iteration: 	100 / 100
Confirmed: 	108
Tentative: 	2
Rejected: 	12

BORUTA RESULTS
Selected features: 108
Rejected features: 14
Tentative features: 2

Top 20 selected features:
   1. ema_ratio_60d_lag1d                                (importance: 0.070235)
   2. bb_width_lag1d                                     (importance: 0.041260)
   3. macd_histogram_lag1d                               (importance: 0.039463)
   4. ema_ratio_20d_lag1d                                (importance: 0.032828)
   5. vol_120d_lag1d                                     (importance: 0.030181)
   6. park

[Parallel(n_jobs=4)]: Done  16 out of  19 | elapsed: 26.5min remaining:  5.0min


Iteration: 	8 / 100
Confirmed: 	0
Tentative: 	122
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	122
Rejected: 	0
Iteration: 	58 / 100
Confirmed: 	115
Tentative: 	4
Rejected: 	3
Iteration: 	58 / 100
Confirmed: 	115
Tentative: 	4
Rejected: 	3
Iteration: 	8 / 100
Confirmed: 	0
Tentative: 	122
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	0
Tentative: 	122
Rejected: 	0
Iteration: 	9 / 100
Confirmed: 	0
Tentative: 	122
Rejected: 	0
Iteration: 	9 / 100
Confirmed: 	0
Tentative: 	122
Rejected: 	0
Iteration: 	59 / 100
Confirmed: 	115
Tentative: 	4
Rejected: 	3
Iteration: 	59 / 100
Confirmed: 	115
Tentative: 	4
Rejected: 	3
Iteration: 	9 / 100
Confirmed: 	0
Tentative: 	122
Rejected: 	0
Iteration: 	9 / 100
Confirmed: 	0
Tentative: 	122
Rejected: 	0
Iteration: 	10 / 100
Confirmed: 	90
Tentative: 	32
Rejected: 	0
Iteration: 	10 / 100
Confirmed: 	90
Tentative: 	32
Rejected: 	0
Iteration: 	10 / 100
Confirmed: 	105
Tentative: 	17
Rejected: 	0
Iteration: 	60 / 100
Confirmed: 	115
Tentative:

[Parallel(n_jobs=4)]: Done  19 out of  19 | elapsed: 29.4min finished
